## 加载所需库并设置用户目录路径


In [ ]:
## Function to run shell commands in Google Colab with R kernel
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...) # Runs the shell command and captures output
  cat(paste0(result, collapse = "\n")) # Prints the output to the console
}

## Function to load multiple R packages
loadPackages = function(pkgs){
  myrequire = function(...){
    suppressWarnings(suppressMessages(suppressPackageStartupMessages(require(...))))
  }
  ok = sapply(pkgs, require, character.only=TRUE, quietly=TRUE) # Checks if each package is installed
  if (!all(ok)){
    message("There are missing packages: ", paste(pkgs[!ok], collapse=", ")) # Prints missing packages
  }
}

## Download the script to add R2U (fast package installation system)
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")
Sys.chmod("add_cranapt_jammy.sh", "0755") # Change permissions to allow execution

## Run the script to set up R2U
shell_call("./add_cranapt_jammy.sh")

## Enable BSPM package manager for installing system-wide packages
bspm::enable()
options(bspm.version.check=FALSE)

## Remove the setup script to keep the workspace clean
shell_call("rm add_cranapt_jammy.sh")

## Define the list of packages to install
cranPkgs2Install = c("dplyr", "ggpubr", "Seurat", "cowplot",
                     "Rtsne", "hdf5r", "patchwork")

## Install all packages without prompting the user
install.packages(cranPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
## To simplify package loading, we created the loadPackages() function. 
## But, if you don't have the function, you should use 'library(name_of_package)'
pkgs = c("Seurat", "dplyr", "patchwork") # List of packages to load
loadPackages(pkgs) # Load packages using previously defined function

## Define the directory where data will be stored and accessed
## IMPORTANT: The user must change "scw01" to match their actual working directory
mydir <- "/content"

# 引言


在本 notebook 中，我们将进入多模态单细胞数据分析，重点关注一个使用 10x Genomics 5' immune profiling 技术处理的非小细胞肺癌（non-small cell lung cancer, NSCLC）样本。该技术能够在单个细胞层面同时捕获 RNA 表达信息和 T 细胞受体（T cell receptor, TCR）序列，从而更完整地刻画样本中的细胞组成与免疫状态。

数据集可从 10x Genomics 网站下载：[NSCLC tumor dataset](https://www.10xgenomics.com/resources/datasets/nsclc-tumor-1-standard-5-0-0)。

* 继续分析之前，请先阅读该网页中关于样本处理和早期分析步骤的详细说明。这些信息有助于理解实验设计、数据生成流程以及数据质量背景。

## TCR 序列

单细胞 TCR 测序可以高分辨率地分析 T 细胞受体多样性，识别特定克隆型（clonotype），并进一步研究它们在免疫系统中的作用。与 bulk 测序把多个细胞的信息混合在一起不同，单细胞方法保留了每个 T 细胞的个体身份，因此可以将转录组状态与 TCR repertoire 关联起来。

#### **注意：**

> 本 notebook 会简要走过单细胞分析的关键步骤。由于本模块的重点是 TCR 序列分析，这里只概览主要流程；如果需要每一步的更详细解释，请参考其他相关模块。

### 读取原始基因计数和元数据


In [ ]:
# Download a filtered gene-barcode matrix from 10X Genomics
# This command uses curl to download a file from the internet. (-O) Saves the file with the same name it has on the server.
shell_call("curl -O https://cf.10xgenomics.com/samples/cell-vdj/5.0.0/vdj_v1_hs_nsclc_multi_5gex_t_b/vdj_v1_hs_nsclc_multi_5gex_t_b_count_filtered_feature_bc_matrix.tar.gz")

## Extract the downloaded file (decompress the dataset)
# tar -xf instructs tar to extract (-x) the specified file (-f) and decompress it.
shell_call("tar -xf /content/vdj_v1_hs_nsclc_multi_5gex_t_b_count_filtered_feature_bc_matrix.tar.gz")

In [ ]:
# Read the filtered feature-barcode matrix into a matrix object
# This command reads the filtered feature-barcode matrix from a 10X Genomics dataset located in the specified 
# directory and assigns the resulting data to the variable counts.
counts <- Read10X(paste0(mydir, "/filtered_feature_bc_matrix/"))

### 创建 Seurat 对象

接下来，我们将使用 count matrix 和 metadata 创建 Seurat 对象。Seurat 对象是一个集中式的数据容器，不仅保存原始计数数据和元数据，也会承载后续分析结果，例如 PCA（principal component analysis，主成分分析）和聚类结果。这样的统一结构便于对 scRNA-seq 数据进行有组织的数据操作和下游分析。

创建 Seurat 对象后，我们可以更高效地管理和分析数据，也更方便执行复杂操作并可视化结果。


In [ ]:
# Create a Seurat object using the raw count matrix
seurat.raw <- CreateSeuratObject(counts = counts)

# Show the contents of Seurat object
seurat.raw

### 探索 Seurat 对象

分析的第一步，是熟悉目前已经存入 Seurat 对象的数据集。Seurat 提供了强大的工具，可以把 count data 与相应 metadata 结合起来探索和可视化，帮助我们全面理解 scRNA-seq 数据。

借助 Seurat，我们可以开展多种探索性分析，例如：

* 检查基因表达分布：可视化不同细胞中的基因表达水平分布，识别高表达基因，并发现潜在离群细胞。

* 探索元数据：查看细胞相关的 metadata，例如细胞类型注释、样本来源和实验条件，从而理解数据集的上下文和样本特征。

* 识别高变基因：寻找在细胞之间表现出显著变异的基因。这些基因通常是聚类、差异表达等下游分析的重要输入。

* 可视化数据：使用 violin plot、feature plot 和 heatmap 等图形探索特定基因的表达模式，并比较不同细胞群之间的差异。

这些初步探索会为后续更深入的分析奠定基础，例如降维、聚类和差异表达分析。这个基础步骤对于把握数据集的细微特征，以及在整个分析流程中作出合理判断都非常关键。


In [ ]:
# How many cells and genes do we currently have?
print(paste0("The number of genes is ", dim(seurat.raw)[1], " and the number of cells is ", dim(seurat.raw)[2]))

# The print() function displays the result in the console.
# The dim() function returns the dimensions of an object, usually a matrix or data frame. 
# In the case of Seurat objects, the gene expression count matrix has genes as rows and cells as columns.
# The paste0() function concatenates (joins) strings without adding any spaces between them.
# The sentences inside the parentheses (") will also be printed

In [ ]:
# View a subset (a slice) of the count matrix 
# Remember: rows are genes, columns are cells/barcodes)
GetAssayData(seurat.raw, layer = "counts")[8:10,13:14]

### **注意：**

> 点号（`.`）表示零值。count table 以 sparse matrix（稀疏矩阵）格式存储；为了节省空间，该格式只显式保存非零值。


In [ ]:
## Display metadata columns available in the Seurat object
# What metadata columns are available in the Seurat object?
print(colnames(seurat.raw@meta.data))

# (@) Accesses the meta.data slot of the seurat.raw object. 
# The meta.data slot typically contains metadata associated with cells, such as cell type, sample ID, or other annotations.
# The colnames() function retrieves the column names of the metadata data frame stored in the meta.data slot

In [ ]:
# Create a violin plot showing the distribution of number of UMIs per cell

options(repr.plot.width=7, repr.plot.height=7) # This command sets the width and height of the plot output.
VlnPlot(seurat.raw, features = c("nCount_RNA"),y.max=2e4) 

# creates a violin plot
# seurat.raw: The Seurat object containing your single-cell data.
# features = c("nCount_RNA"): It is plotting the number of RNA molecules (nCount_RNA) for each cell.
# y.max=2e4: Sets the maximum value for the y-axis to 20,000 (2e4 is scientific notation for 20,000)

### 质量控制

前面已经看到，Seurat 会自动计算两个主要质量控制（quality control, QC）指标，例如 UMI（unique molecular identifier）数量和每个细胞检测到的基因数。接下来还需要关注的一个重要 QC 指标是线粒体基因比例。线粒体基因表达升高可能提示细胞应激或凋亡，因此有必要监测其表达水平。

我们将使用 Seurat 的 `PercentageFeatureSet` 方法计算线粒体基因比例。该方法会计算来自某一指定基因模式的 UMI 在总 UMI 中所占的百分比。对于线粒体基因，通常查找以 `MT-` 开头的基因名，这是常见的线粒体基因前缀。


In [ ]:
# Human mitochondrial gene names start with "MT-" so we'll calculate the percentage of genes matching the pattern "^MT-"
# This function calculates the percentage of counts for features (genes) that match a given pattern.
# Also creates a new metadata column in the seurat.raw object named percent.mt and assigns the calculated percentages to it. (seurat.raw[["percent.mt"]])
seurat.raw[["percent.mt"]] <- PercentageFeatureSet(seurat.raw, pattern = "^MT-") 

# Now we can see that the % mitochondrial gene expression has been calculated for each cell
head(seurat.raw$percent.mt) # Display the first few rows of the calculated mitochondrial percentages

In [ ]:
## Violin plot for three quality metrics: UMI count, gene count, mitochondrial gene percentage
options(repr.plot.width=12, repr.plot.height=6)
VlnPlot(seurat.raw, features = c("nCount_RNA", "nFeature_RNA", "percent.mt")) 

# We can visualize all three of the cell quality metrics together using Seurat's VlnPlot method

把这些 QC 指标联合可视化通常很有帮助，因为在多个维度上同时离群的细胞更可能是低质量细胞。Seurat 的 `FeatureScatter` 可以基于 metadata 中指定的两列绘制散点图。


In [ ]:
## Scatter plot: total UMI counts vs percentage of mitochondrial genes
options(repr.plot.width=6, repr.plot.height=6)
FeatureScatter(seurat.raw, feature1 = "nCount_RNA", feature2 = "percent.mt") 

# Here we visualize the number of UMI vs % mito genes

In [ ]:
# Once we've visualized the metrics we can select the thresholds that we want to use to filter.
# We use R's subset method to filter
seurat.raw <- subset(
    seurat.raw,
    subset = ## Apply filtering criteria:
        nFeature_RNA > 200 &  #  Remove cells with fewer than 200 genes
        nCount_RNA > 400 &    # Remove cells with fewer than 400 UMIs
        nFeature_RNA < 6000 & # Remove cells with more than 6000 genes (potential doublets)
        percent.mt < 40)      # Remove cells with more than 40% mitochondrial RNA (low-quality cells)


### 在 Seurat 中归一化数据

从数据集中去除不需要的细胞后，下一步是对数据进行归一化。归一化用于校正不同细胞之间测序深度的差异，使细胞之间的表达水平比较更有意义。

1. 归一化方法：LogNormalize

    * Seurat 中最常用的归一化方法是称为 `LogNormalize` 的全局缩放归一化方法，主要包含三个步骤。

2. 按总表达量归一化

    * 对每个细胞，先计算总表达量，即所有基因 count 的总和。

    * 然后将每个基因的表达值除以该细胞的总表达量，以校正测序深度差异。

3. 乘以缩放因子

    * 归一化后的值再乘以一个缩放因子，默认值为 10,000。这样可以把数值调整到更方便解释的尺度。

4. 对数转换

    * 最后，对归一化后的值进行自然对数转换。log 转换可以稳定方差、降低离群值影响，并使数据分布更接近对称。

5. 数据存储

    * 原始 raw counts 存储在 `seurat.raw[["RNA"]]@counts` 中。该 slot 保存每个细胞中每个基因的未归一化 count 数据。

    * 新生成的归一化数据存储在 `seurat.raw[["RNA"]]@data` 中。该 slot 保存每个细胞中每个基因的归一化并 log 转换后的表达值。


In [ ]:
# Normalize data using LogNormalization
seurat.raw <- NormalizeData(seurat.raw, normalization.method = "LogNormalize", scale.factor = 10000)

## 执行标准流程：高变基因识别、缩放、PCA、聚类和 UMAP


In [ ]:
# Identify the most variable genes
seurat.raw <- FindVariableFeatures(seurat.raw, selection.method = "vst", nfeatures = 2000)
# Scale the data
seurat.raw <- ScaleData(seurat.raw, features = VariableFeatures(seurat.raw), do.scale = T, do.center = T)
# Run PCA (Principal Component Analysis)
seurat.raw <- RunPCA(seurat.raw, features = VariableFeatures(seurat.raw))
# Find cell neighbors
seurat.raw <- FindNeighbors(seurat.raw, dims = 1:20, k.param = 20)
# Identify clusters at low resolution
seurat.raw <- RunUMAP(seurat.raw, dims = 1:20, reduction = "pca", seed.use = 1)

# Find clusters and use a low resolution (0.1 is a good start) so that we can easily identify all of the T cells later
seurat.raw <- FindClusters(seurat.raw, resolution = 0.1)

## Visualize UMAP with cluster labels
DimPlot(seurat.raw, reduction = "umap", label = T,group.by = "seurat_clusters")

查看常见细胞类型 marker 的 feature plot，判断哪些 cluster 是 T 细胞。

* T 细胞基因：CD3D、CD8A、GNLY
* B 细胞基因：CD79A
* 髓系细胞基因：FCGR3A
* 上皮细胞（肺）基因：KRT7


In [ ]:
# Feature plot of specific marker genes to identify cell types
FeaturePlot(seurat.raw, features = c("CD3D", "CD8A", "GNLY", "CD79A", "FCGR3A","KRT7"), min.cutoff = "q1")

### TCR 序列整合

在本节中，我们将读取 10x Cell Ranger 软件生成的文件。这些文件描述了在单个细胞中检测到的 TCR 序列。


In [ ]:
#Download files with curl
shell_call("curl -O https://cf.10xgenomics.com/samples/cell-vdj/5.0.0/vdj_v1_hs_nsclc_multi_5gex_t_b/vdj_v1_hs_nsclc_multi_5gex_t_b_vdj_t_all_contig_annotations.csv")
shell_call("curl -O https://cf.10xgenomics.com/samples/cell-vdj/5.0.0/vdj_v1_hs_nsclc_multi_5gex_t_b/vdj_v1_hs_nsclc_multi_5gex_t_b_vdj_t_clonotypes.csv")

# Read in TCR information for each cell
tcr <- read.csv(paste0(mydir,"/vdj_v1_hs_nsclc_multi_5gex_t_b_vdj_t_all_contig_annotations.csv"))

# Read in clonotype  info 
# many cells can share the same clonotype, or TCR sequence)
clono <- read.csv(paste0(mydir,"/vdj_v1_hs_nsclc_multi_5gex_t_b_vdj_t_clonotypes.csv"))

# Remove the -1 at the end of each barcode.
tcr$barcode <- gsub("-1", "", tcr$barcode)

# Filter to keep only the first line of each cell barcode
# so that we ony analyze one clonotype for each cell.
tcr <- tcr[!duplicated(tcr$barcode), ]

# Also remove the -1 at the end of the line of the Seurat object
seurat.raw = RenameCells(seurat.raw,new.names = gsub("-1", "", colnames(seurat.raw)))

# Only keep the barcode and clonotype columns. 
tcr <- tcr[,c("barcode", "raw_clonotype_id")]

# Adjust the name from "raw_clonotype_id" to "clonotype_id" so
names(tcr)[names(tcr) == "raw_clonotype_id"] <- "clonotype_id"

# Merge the TCR and clonotype tables so we have TCR amino acid sequence for each cell
tcr <- merge(tcr, clono[, c("clonotype_id", "cdr3s_aa")])

# Reorder so barcodes are first column
tcr <- tcr[, c(2,1,3)]

# Set them as rownames
rownames(tcr) <- tcr[,1]

#  Remove the unnecessary extra column of barcodes
tcr[,1] <- NULL

# Add to the Seurat object's metadata.
seurat.raw <- AddMetaData(object=seurat.raw, metadata=tcr)

# Confirm we have TCR information in the metadata
head(seurat.raw@meta.data)


In [ ]:
# Generate a histogram of clone frequencies from clono table
# What is a good threshold to distinguish expanded from non-expanded clones?
barplot(table(clono$frequency),xlab="Number of cells in clone") 

# The barplot() function creates a bar plot.
# The table() function creates a contingency table of the counts at each unique value of clono$frequency.
# clono$frequency accesses the frequency column in the clono data frame


# Identity expaned clones, flag in metadata, and label in UMAP
# How many expanded clones are there? Try changing the threshold and compare the results
Nexpand = 1  
# This is the threshold we will use to distinguish expanded from non-expanded clones
length(which(clono$frequency>1))
# which(clono$frequency > 1): Returns the indices of the rows where the frequency is greater than 1.
# length(which(clono$frequency > 1)): Counts the number of elements (clones) that have a frequency greater than 1. This gives you the number of expanded clones.

expanded_clones = clono$clonotype_id[clono$frequency>1] 
# clono$clonotype_id[clono$frequency > 1]: Selects the clonotype_id values where the corresponding frequency is greater than 1.
#expanded_clones: Stores the clonotype IDs of the expanded clones in a new variable.


# Add a new metadata column indicating which cells are part of an expanded clone
seurat.raw = AddMetaData(seurat.raw,metadata = rep("no",ncol(seurat.raw)),col.name="TCR_expanded") 
# This command adds a new metadata column to the seurat.raw Seurat object.
# in this new metadata where every cell is "no", and then changing the value for the expanded cells to "yes"

seurat.raw@meta.data$TCR_expanded[seurat.raw@meta.data$clonotype_id %in% expanded_clones] = "yes" 
# This command updates the TCR_expanded metadata column.
# seurat.raw@meta.data$TCR_expanded: Accesses the TCR_expanded column in the metadata.
# seurat.raw@meta.data$clonotype_id %in% expanded_clones: Checks which cells have clonotype_id values that are in the expanded_clones list.
# = "yes": Sets the value to "yes" for cells that belong to expanded clones

# Look at the UMAP to see if expanded clones have similar gene expression
DimPlot(seurat.raw, reduction = "umap", group.by = "TCR_expanded", label = T)

# Do expanded T cells have different genes expressed compared to non-expanded T cells?
# To do ask this question, let's first separate the T cells from the rest of the object
# Check your clustering to see which one looks like it contains T cells

Idents(seurat.raw) = "seurat_clusters" # This means that subsequent operations will use the cluster identities assigned to each cell to seurat_clusters
seurat.t = subset(seurat.raw, idents = "1") #is the new Seurat object containing only the cells from cluster 1.
seurat.t # Check the number of samples (cells) to see how many T cells we have to work with

Idents(seurat.t) = "TCR_expanded" # This command sets the active identity class in the seurat.t object to TCR_expanded
deg_expanded = FindMarkers(seurat.t,ident.1="yes",ident.2="no",logfc.threshold = 0.25,min.pct = 0.1)
# This command identifies differentially expressed genes between two groups of cells
# Those marked as "yes" in the TCR_expanded metadata and those marked as "no".
# logfc.threshold: Minimum log2 fold-change threshold for identifying differentially expressed genes.
# min.pct: Minimum percentage of cells in which the gene is detected.


# Visualize the expression of the top DE genes for each annotated cell subset.
top30genes <- deg_expanded %>% filter(avg_log2FC > 0) %>% top_n(30, avg_log2FC)
# deg_expanded %>% filter(avg_log2FC > 0): This filters the differentially expressed genes (deg_expanded) 
# to include only those with a positive average log2 fold-change (avg_log2FC > 0).
# top_n(30, avg_log2FC): From the filtered genes, this selects the top 30 genes with the highest average log2 fold-change.

genes <- rownames(top30genes)
seurat.t <- ScaleData(seurat.t, features = genes, do.center = T, do.scale = T)
# Scales the expression data for the selected genes (features = genes) in the seurat.t object.
# do.center = T: Centers the data by subtracting the mean expression for each gene.
# do.scale = T: Scales the data by dividing by the standard deviation for each gene


DoHeatmap(seurat.t, features = genes) 
# Creates a heatmap of the expression data
